# 🧠 Notebook 01: 3D VisReg JEPA Training & Low-Data Benchmark
### Part of the BraTS 3D Volumetric Multimodal JEPA Research Suite

This dedicated notebook runs the **3D VisReg JEPA** pipeline:
1. **Self-Supervised Pre-training (50 Epochs, AMP)**: Single-encoder JEPA with decoupled Center, Scale (unit variance penalty), and 1D Sliced-Wasserstein Shape Regularization against standard normal quantiles.
2. **Downstream Volumetric Fine-Tuning (30 Epochs, AMP)**: Multi-Scale 3D FPN decoder bridging latent tokens ($L_2, L_4, L_6, L_8$) to dense $128^3$ voxel masks.
3. **Full-Data Held-Out Test Evaluation**: Exact 3D Dice, IoU, 95th Percentile Hausdorff Distance (mm), and inference latency.
4. **Low-Data Volumetric Label Efficiency**: Benchmarks fine-tuning from $1\%$ to $100\%$ labels ($13$ to $1,266$ patient volumes).
5. **Artifact Export**: Bundles checkpoints and metrics into `visreg_outputs.zip`.

> **Estimated Runtime**: ~3.0 - 3.5 hours on NVIDIA Tesla T4 GPU (Well within Kaggle's 12-hour session limit).


## 1. Hardware & CUDA Environment Verification


In [ ]:
import datetime
import time

NOTEBOOK_START_TIME = time.time()
NOTEBOOK_START_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"⏱️ Session Start Time: {NOTEBOOK_START_STR}")

!nvidia-smi

import torch

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(
        "WARNING: No GPU detected. Please navigate to Notebook Settings -> Accelerator -> GPU T4!"
    )

## 2. Dependencies Installation


In [ ]:
!pip install -q --no-cache-dir monai nibabel tabulate matplotlib
import monai
import nibabel as nib
import tabulate

print(f"✓ MONAI Version:    v{monai.__version__}")
print(f"✓ NiBabel Version:  v{nib.__version__}")
print(f"✓ Tabulate Version: v{tabulate.__version__}")

## 3. Codebase Setup & Editable Installation


In [ ]:
import os
import shutil
import sys
from pathlib import Path

# Setup working directory in /kaggle/working
REPO_URL = "https://github.com/hanriman/tumor_segmentation_3d.git"
work_dir = Path("/kaggle/working/tumor_segmentation_3d")

# Clean up broken or incomplete clone from previous failed runs
if work_dir.exists() and not (work_dir / "src" / "brats_jepa_3d").exists():
    print("⚠️ Cleaning up incomplete repository clone...")
    shutil.rmtree(work_dir)

thesis_repo = Path("/kaggle/working/thesis_repo")
if thesis_repo.exists() and not (thesis_repo / "src" / "brats_jepa_3d").exists():
    shutil.rmtree(thesis_repo)

# Clone repository if not already present
if not (work_dir / "src" / "brats_jepa_3d").exists():
    if (thesis_repo / "src" / "brats_jepa_3d").exists():
        work_dir = thesis_repo
    elif Path("/kaggle/working/src/brats_jepa_3d").exists():
        work_dir = Path("/kaggle/working")
    else:
        print(f"Cloning codebase from: {REPO_URL} ...")
        !git clone {REPO_URL} {work_dir}

# Verify package was successfully cloned
src_dir = work_dir / "src"
if not (src_dir / "brats_jepa_3d").exists():
    raise RuntimeError(
        "❌ Clone failed! The package 'brats_jepa_3d' was not found on disk.\n"
        "👉 Please ensure 'Internet' is toggled ON in the Kaggle notebook settings (right sidebar)!"
    )

# Change working directory and update sys.path
os.chdir(str(work_dir))
%cd {work_dir}

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Install in editable mode
!pip install -q -e .

print(f"\n✓ Working directory set to: {Path.cwd()}")
!git log -1 --oneline 2>/dev/null || echo "(Git commit info unavailable)"


## 4. Dataset Discovery & Health Checks
Verifies whether processed `.npz` volumes exist. If absent, automatically invokes `prepare_data_3d.py` with multi-worker parallel resampling (`--num_workers 4`) and compact `float16` storage (~6.7 MB per volume, upcast to FP32 in RAM upon loading).


In [ ]:
import pandas as pd
import torch

from brats_jepa_3d.config import get_dataset_dir, get_metadata_path
from brats_jepa_3d.data import BraTS3DDataset

print("=== DATASET DISCOVERY ===")
data_dir = get_dataset_dir("brats_gli_3d")
meta_path = get_metadata_path("brats_gli_3d")
print(f"Dataset Directory: {data_dir} (Exists: {data_dir.exists()})")
print(f"Metadata CSV:      {meta_path} (Exists: {meta_path.exists()})")

# If preprocessed dataset is not found, check for raw BraTS data and run prepare_data_3d.py
if not meta_path.exists():
    print("\n⚠️ Preprocessed dataset not found. Running 3D preprocessing from raw BraTS data...")
    !python scripts/prepare_data_3d.py --limit 100 --dtype float16 --num_workers 4
    meta_path = get_metadata_path("brats_gli_3d")

if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"\n✓ Loaded Metadata: {len(df)} total records across splits:")
    print(df["split"].value_counts().to_string())

    ds = BraTS3DDataset(split="train")
    sample = ds[0]
    print("\n✓ Sample Tensor Verification:")
    print(f"  Image Shape: {sample['image'].shape} (dtype: {sample['image'].dtype})")
    print(f"  Mask Shape:  {sample['mask'].shape} (dtype: {sample['mask'].dtype})")
    print(f"  Tumor Voxels: {int((sample['mask'] > 0).sum()):,}")
else:
    print(
        "❌ ERROR: Please attach 'brats-3d-datasets' or raw BraTS dataset via '+ Add Input' in Kaggle!"
    )


## 5. Phase 1: 3D VisReg JEPA Self-Supervised Pre-training (50 Epochs, AMP)
Trains the 3D Vision Transformer backbone exclusively in latent feature space using decoupled optimal transport Sliced-Wasserstein regularization.

*Note: For a 2-minute smoke test, append `--smoke_test`.*


In [ ]:
!python scripts/train_jepa_3d.py \
    --model_type visreg_jepa \
    --epochs 50 \
    --batch_size 8 \
    --num_workers 2 \
    --learning_rate 5e-4 \
    --weight_decay 0.05 \
    --amp

## 6. Phase 2: Downstream Volumetric Fine-Tuning with Multi-Scale 3D FPN
Couples the pre-trained 3D VisReg encoder with a hierarchical 3D Feature Pyramid Network (FPN) decoder fusing representations at $8^3 \to 16^3 \to 32^3 \to 64^3 \to 128^3$.


In [ ]:
# Identify the pre-trained checkpoint
from pathlib import Path
from brats_jepa_3d.config import CHECKPOINTS_DIR
from brats_jepa_3d.utils import sort_checkpoints_by_epoch

epoch_ckpts = sort_checkpoints_by_epoch(list(CHECKPOINTS_DIR.glob("visreg_jepa*epoch*.pt")))
best_ckpts = sorted(CHECKPOINTS_DIR.glob("visreg_jepa*best.pt"))
ckpt_candidates = epoch_ckpts or best_ckpts
pretrained_ckpt = (
    str(ckpt_candidates[-1]) if ckpt_candidates else str(CHECKPOINTS_DIR / "visreg_jepa_best.pt")
)
print(f"Using pre-trained checkpoint: {pretrained_ckpt}")

!python scripts/train_downstream_3d.py \
    --model_type visreg_jepa \
    --decoder_type multiscale \
    --pretrained_checkpoint {pretrained_ckpt} \
    --epochs 30 \
    --batch_size 2 \
    --num_workers 2 \
    --learning_rate 3e-4 \
    --amp


## 7. Phase 3: Full-Data Held-Out Test Split Evaluation
Evaluates the trained 3D VisReg JEPA segmentation model across all $271$ held-out test volumes.


In [ ]:
!python scripts/evaluate_3d.py \
    --model_type visreg_jepa \
    --decoder_type multiscale \
    --num_workers 2 \
    --amp

## 8. Phase 4: Low-Data Volumetric Label Efficiency Benchmark
Evaluates fine-tuning performance across annotation budgets: $1\%$ ($13$ vols), $5\%$ ($63$ vols), $10\%$ ($127$ vols), $25\%$ ($316$ vols), $50\%$ ($633$ vols), and $100\%$ ($1,266$ vols).


In [ ]:
!python scripts/evaluate_low_data_3d.py \
    --model_type visreg_jepa \
    --fractions 0.01 0.05 0.10 0.25 0.50 1.00 \
    --epochs 15 \
    --batch_size 2 \
    --num_workers 2 \
    --amp

## 9. Phase 5: Export & Package Artifacts
Packages all trained weights, training logs, and metric JSON files into `visreg_outputs.zip` for 1-click download and chaining into Notebook 04.


In [ ]:
!mkdir -p /kaggle/working/export_visreg
!cp -r outputs/checkpoints /kaggle/working/export_visreg/ 2>/dev/null || true
!cp -r outputs/metrics /kaggle/working/export_visreg/ 2>/dev/null || true
!cp -r outputs/logs /kaggle/working/export_visreg/ 2>/dev/null || true

!cd /kaggle/working && zip -r -q visreg_outputs.zip export_visreg/
print("✓ visreg_outputs.zip ready for download!")
!ls -lh /kaggle/working/visreg_outputs.zip

import datetime
import time

NOTEBOOK_END_TIME = time.time()
NOTEBOOK_END_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
start_time = globals().get("NOTEBOOK_START_TIME", NOTEBOOK_END_TIME)
total_elapsed_sec = NOTEBOOK_END_TIME - start_time
hours, rem = divmod(total_elapsed_sec, 3600)
minutes, seconds = divmod(rem, 60)

print("
" + "=" * 50)
print(f"⏱️ Session Start Time:   {globals().get('NOTEBOOK_START_STR', 'N/A')}")
print(f"⏱️ Session End Time:     {NOTEBOOK_END_STR}")
print(f"⏱️ Total Execution Time: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({total_elapsed_sec:.2f}s)")
print("=" * 50)
